In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

In [2]:
import os
import math
import numpy as np
import torch
from tqdm.auto import tqdm
from datetime import timedelta
import time
import gc

from transformers import (
    DataCollatorForLanguageModeling,
    TrainerCallback,
)
from datasets import load_dataset, Dataset

from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments

torch.backends.cudnn.benchmark = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Consider adjusting PYTORCH_CUDA_ALLOC_CONF based on errors, but expandable_segments is often helpful
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # Keep if it helps prevent OOMs
torch.cuda.empty_cache()
gc.collect()

# --- Configuration ---
# Using a 4bit pre-quantized model optimized by Unsloth
base_model_name = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit"
output_dir = "./qwen_math_nbody_lora_unsloth_continued_pt" # More specific name
num_epochs_to_train = 5
chunk_size = 2048 # Corresponds to max_seq_length for the model
validation_size = 1200
# Note: 5e-6 is a relatively low LR for LoRA. Standard LoRA often uses 1e-4 to 5e-5.
# However, for continued pretraining, a lower LR might be desired to adapt gently. Monitor loss.
unsloth_learning_rate = 5e-6

# --- Determine Dtype and 4bit Loading ---
# Unsloth's default dtype=None handles auto-detection, but explicit check is also fine.
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    print("GPU supports BF16, using bfloat16.")
    model_dtype = torch.bfloat16
    # Unsloth automatically enables TF32 on Ampere+ GPUs for speed gains.
else:
    print("GPU does not support BF16 or is older, using float16.")
    model_dtype = torch.float16
load_in_4bit_flag = True # Must be True since we're using a "-bnb-4bit" model

# --- Model Initialization with Unsloth ---
print(f"Initializing Unsloth FastLanguageModel: {base_model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = chunk_size,       # Unsloth handles RoPE scaling internally if needed
    dtype = model_dtype,               # Determined above (bf16 or fp16)
    load_in_4bit = load_in_4bit_flag,  # Use 4-bit quantization
    # token = "hf_...", # Add your Hugging Face token if needed (e.g., for Llama models)
    # trust_remote_code=True # Usually not needed for Unsloth's official models
)
print("Base model and tokenizer loaded via Unsloth.")

# --- Apply LoRA using Unsloth's Method ---
print("Applying LoRA config to model via Unsloth...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank of the LoRA matrices. Guideline suggests 8, 16, 32, 64, 128.
    lora_alpha = 32, # Scaling factor for LoRA. Often 2*r.
    # Target modules for Qwen2.5/Llama style models. Ensure these match your model architecture if different.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # Unsloth Recommendation: lora_dropout = 0 is optimized for speed and memory.
    lora_dropout = 0,
    # Unsloth Recommendation: bias = "none" is optimized.
    bias = "none",
    # Unsloth specific optimization for gradient checkpointing - VERY IMPORTANT for memory saving.
    use_gradient_checkpointing = "unsloth",
    random_state = 3407, # For reproducibility
    use_rslora = False,  # Rank Stabilized LoRA (another technique, False by default)
    loftq_config = None, # LoftQ initialization (another technique, None by default)
    # modules_to_save = None # Not saving/training extra modules like lm_head/embed_tokens for pure LoRA
)
print("LoRA applied via Unsloth.")
model.print_trainable_parameters() # Shows the percentage of trainable parameters (should be small for LoRA)

print("\nLoading previously trained adapter weights (from book training)...")
# Define the path where the final book adapter was saved in the *previous* run
previous_adapter_path = "/kaggle/input/book-adapter-unsloth/pytorch/default/1"

if os.path.exists(previous_adapter_path):
    try:
        # Use the PEFT method to load adapter weights into the existing LoRA layers
        model.load_adapter(previous_adapter_path)
        print(f"Successfully loaded adapter weights from: {previous_adapter_path}")
    except Exception as e:
        print(f"Error loading adapter from {previous_adapter_path}: {e}")
        print("Proceeding with potentially uninitialized LoRA weights (check path and previous run).")
        # Depending on your needs, you might want to raise an error here instead
        # raise RuntimeError(f"Failed to load previous adapter: {e}")
else:
    print(f"Warning: Previous adapter path not found: {previous_adapter_path}")
    print("Proceeding with potentially uninitialized LoRA weights (this will not continue training).")
    # Consider raising an error if continuing is critical
    # raise FileNotFoundError(f"Previous adapter path not found: {previous_adapter_path}")
print("-" * 40)
# <<< --- END ADAPTER LOADING BLOCK --- >>>

# --- Set Pad Token ---
# Necessary for Causal LM if tokenizer doesn't have one. EOS token is common practice.
if tokenizer.pad_token is None:
    print("Setting pad token to eos token.")
    # Check if EOS token exists. It *should* for most models.
    if tokenizer.eos_token is None:
        raise ValueError("Tokenizer does not have an EOS token to use as PAD token.")

    original_vocab_size = len(tokenizer)
    tokenizer.pad_token = tokenizer.eos_token

    # Important: Resize model embeddings ONLY if the pad token was truly added and wasn't already EOS.
    # In most cases where pad_token is None, setting it to eos_token doesn't change the vocab size
    # because eos_token already exists. Double check if resize is actually needed.
    if len(tokenizer) > original_vocab_size:
         print(f"Resizing model embeddings from {original_vocab_size} to {len(tokenizer)} due to added pad token...")
         # Ensure the model object is the one with PEFT applied if resizing needed AFTER get_peft_model
         model.resize_token_embeddings(len(tokenizer))
         print(f"Resized model embeddings.")
    else:
         print("Pad token set to existing EOS token. No embedding resize needed.")


# --- Dataset Loading ---
print("Loading dataset...")
# Ensure this path is correct in your environment
data_file_path = '/kaggle/input/paper-batch/cleaned_batch1.md'
if not os.path.exists(data_file_path):
    raise FileNotFoundError(f"Dataset file not found: {data_file_path}")
try:
    with open(data_file_path, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()
    print(f"Loaded {len(all_lines)} lines from dataset.")
except Exception as e:
    raise IOError(f"Error reading dataset file {data_file_path}: {e}")


# --- Dataset Preparation for Continued Pretraining ---
if not all_lines:
     raise ValueError("Dataset file is empty.")
if len(all_lines) <= validation_size:
     raise ValueError(f"validation_size ({validation_size}) is >= total lines ({len(all_lines)}). Need more data or smaller validation set.")

train_lines = all_lines[:-validation_size]
validation_lines = all_lines[-validation_size:]
print(f"Splitting dataset: {len(train_lines)} training lines, {len(validation_lines)} validation lines.")

# Combine lines into single strings for tokenization
train_corpus = "".join(train_lines)
validation_corpus = "".join(validation_lines)

# Function to tokenize and chunk raw text for Causal LM pretraining/continued pretraining
def prepare_corpus_for_training(corpus, tokenizer, chunk_size, dataset_name=""):
    """Tokenizes and chunks the corpus for Causal LM. Labels are same as input_ids."""
    if not corpus:
        print(f"Warning: Corpus for {dataset_name} is empty. Skipping.")
        return None

    print(f"Tokenizing {dataset_name} corpus (this may take a while for large datasets)...")
    # Use `return_attention_mask=False` if mask is not explicitly needed later, might save memory/time
    tokens_dict = tokenizer(corpus, truncation=False, add_special_tokens=True, return_attention_mask=False)
    tokens = tokens_dict["input_ids"]
    print(f"Total tokens in {dataset_name}: {len(tokens)}")

    if not tokens:
        print(f"Warning: No tokens generated for {dataset_name}. Check corpus content.")
        return None

    total_expected_chunks = math.ceil(len(tokens) / chunk_size)
    print(f"Creating approximately {total_expected_chunks} chunks of size {chunk_size} for {dataset_name}...")

    chunks = []
    # Using list comprehension for potential slight speedup if memory allows
    # for i in tqdm(range(0, len(tokens), chunk_size), desc=f"Chunking {dataset_name} Data"):
    #     chunk_tokens = tokens[i : i + chunk_size]
    #     # Pad the last chunk if it's smaller than chunk_size
    #     if len(chunk_tokens) < chunk_size:
    #         padding_length = chunk_size - len(chunk_tokens)
    #         chunk_tokens = chunk_tokens + [tokenizer.pad_token_id] * padding_length
    #     # For Causal LM, labels are usually the same as input_ids (shifted internally by model/trainer)
    #     labels = chunk_tokens[:] # Use slicing to ensure it's a copy
    #     chunks.append({"input_ids": chunk_tokens, "labels": labels})

    # Alternative: Generator approach might be more memory efficient for huge datasets, but requires yielding dicts
    # This standard loop with tqdm is clear and usually fine.
    for i in tqdm(range(0, len(tokens), chunk_size), desc=f"Chunking {dataset_name} Data", total=total_expected_chunks):
        chunk_start = i
        chunk_end = i + chunk_size
        chunk_tokens = tokens[chunk_start : chunk_end]

        # Pad the last chunk if necessary
        if len(chunk_tokens) < chunk_size:
            padding_length = chunk_size - len(chunk_tokens)
            # Ensure pad_token_id exists
            if tokenizer.pad_token_id is None:
                 raise ValueError("Tokenizer pad_token_id is None. Cannot pad sequences.")
            chunk_tokens.extend([tokenizer.pad_token_id] * padding_length)

        # For Causal LM pretraining, labels are the input_ids
        labels = chunk_tokens[:] # Create a copy

        chunks.append({"input_ids": chunk_tokens, "labels": labels})


    if not chunks:
        print(f"Warning: No chunks were created for {dataset_name}. Check tokenization output and chunk_size.")
        return None

    print(f"Created {len(chunks)} chunks for {dataset_name}.")
    # Create Hugging Face Dataset object
    try:
        return Dataset.from_list(chunks)
    except Exception as e:
        print(f"Error creating Dataset object for {dataset_name}: {e}")
        return None


print("Preparing training dataset...")
train_dataset = prepare_corpus_for_training(train_corpus, tokenizer, chunk_size, dataset_name="Training")
if train_dataset:
     print(f"Training dataset size: {len(train_dataset)} chunks")
     # Optional: Inspect a sample
     # print("Sample Train chunk:", train_dataset[0])
else:
     raise ValueError("Failed to create training dataset. Check logs.")

print("Preparing validation dataset...")
validation_dataset = prepare_corpus_for_training(validation_corpus, tokenizer, chunk_size, dataset_name="Validation")
if validation_dataset:
    print(f"Validation dataset size: {len(validation_dataset)} chunks")
    # Optional: Inspect a sample
    # print("Sample Validation chunk:", validation_dataset[0])
else:
    print("Warning: Validation dataset could not be created or is empty. Proceeding without evaluation.")
    validation_dataset = None # Explicitly set to None for clarity

# --- Data Collator ---
# Standard for Causal LM: Takes care of batching and potentially padding (though our data is pre-padded)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # IMPORTANT: False for Causal Language Modeling (like GPT, Llama, Qwen)
)

# --- Training Arguments using Unsloth ---
print("Defining Unsloth Training Arguments...")
training_args = UnslothTrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=num_epochs_to_train,
    # Adjust batch size and gradient accumulation based on GPU memory
    per_device_train_batch_size=4, # Small batch size due to long sequence length & potential memory limits
    per_device_eval_batch_size=4,  # Can often be slightly larger than train batch size
    gradient_accumulation_steps=8, # Increase effective batch size (2 * 32 = 64 effective batch size)
    learning_rate=unsloth_learning_rate,
    # embedding_learning_rate = None, # NOT USED unless `modules_to_save` includes embedding/lm_head
    weight_decay=0.01, # Standard value
    logging_steps=3,  # Log metrics every 10 steps
    save_strategy="epoch", # Save a checkpoint at the end of each epoch
    save_total_limit=2,    # Keep only the last 2 checkpoints + final model
    dataloader_drop_last=True, # Drop last incomplete batch to ensure consistent shapes
    # fp16 / bf16 are enabled AUTOMATICALLY by Unsloth based on model dtype and hardware. No need to set here.
    logging_first_step=True, # Log metrics at the very first step
    logging_dir=f"{output_dir}/logs", # Directory for TensorBoard logs
    warmup_ratio=0.1, # Use 10% of total steps for learning rate warmup
    lr_scheduler_type="cosine", # Common learning rate scheduler
    optim="adamw_8bit",
    max_grad_norm=0.5, # Gradient clipping to prevent exploding gradients
    # Consider increasing dataloader_num_workers if data loading is a bottleneck (requires more RAM)
    # Evaluation strategy depends on whether validation_dataset exists
    eval_strategy="epoch" if validation_dataset else "no",
    # Load the best model found during training (based on eval_loss) at the end. Requires eval_strategy != "no"
    load_best_model_at_end=True if validation_dataset else False,
    metric_for_best_model="eval_loss" if validation_dataset else None,
    greater_is_better=False, # For loss, lower is better
    # gradient_checkpointing=True, # DO NOT SET HERE - Handled by `use_gradient_checkpointing="unsloth"` in get_peft_model
    seed=42, # For reproducibility
    report_to=["tensorboard"], # Logging integrations (can add "wandb")
    # push_to_hub=False, # Set to True to push adapter to Hugging Face Hub
    # hub_model_id="your_username/your_model_id", # Required if push_to_hub=True
    # resume_from_checkpoint=False, # Set to True or path to resume training
)

# --- Trainer Initialization using Unsloth ---
print("Initializing UnslothTrainer...")
trainer = UnslothTrainer(
    model=model, # The Unsloth model with LoRA adapters
    args=training_args, # UnslothTrainingArguments
    train_dataset=train_dataset,
    eval_dataset=validation_dataset, # Pass None if no validation set
    data_collator=data_collator,
    tokenizer=tokenizer
)

# --- Start Training ---
# Calculate effective batch size for logging
try:
    # Get number of devices from accelerator if available (multi-GPU setup)
    num_processes = trainer.accelerator.num_processes
except AttributeError:
    num_processes = 1 # Assume single device if accelerator not fully initialized or not used

effective_batch_size = (
    trainer.args.per_device_train_batch_size *
    trainer.args.gradient_accumulation_steps *
    num_processes
)
print(f"\nEffective batch size: {trainer.args.per_device_train_batch_size} (per_device) * "
      f"{trainer.args.gradient_accumulation_steps} (accumulate) * "
      f"{num_processes} (devices) = {effective_batch_size}")

print(f"Max Sequence Length (chunk_size): {chunk_size}")
print(f"Number of training examples: {len(train_dataset) if train_dataset else 0}")
print(f"Number of validation examples: {len(validation_dataset) if validation_dataset else 0}")
print(f"Number of training epochs: {training_args.num_train_epochs}")
print(f"Optimizer: {training_args.optim}")
print(f"Learning Rate: {training_args.learning_rate}")
print(f"{'*'*70}\n")


print(f"\n🚀 Starting LoRA continued pretraining for {trainer.args.num_train_epochs} epochs using Unsloth...\n")
training_start_time = time.time()
try:
    # Use resume_from_checkpoint=True in args or pass path here if needed
    train_result = trainer.train(resume_from_checkpoint=training_args.resume_from_checkpoint)
    print("\n✅ Training completed successfully!")

    training_end_time = time.time()
    print(f"Total Training Time: {timedelta(seconds=int(training_end_time - training_start_time))}")

    # --- Save Final Adapter ---
    # If load_best_model_at_end=True, this saves the best adapter. Otherwise, it saves the last one.
    final_adapter_path = os.path.join(output_dir, "final_adapter")
    print(f"Saving final LoRA adapter to {final_adapter_path}...")
    # Unsloth's FastLanguageModel object supports save_pretrained for PEFT adapters
    model.save_pretrained(final_adapter_path)
    # Important: Save the tokenizer along with the adapter
    tokenizer.save_pretrained(final_adapter_path)
    print(f"Adapter and tokenizer saved successfully to {final_adapter_path}")

    # Log final metrics
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    trainer.save_state() # Saves optimizer state, scheduler state, etc.

except KeyboardInterrupt:
    print("\n❌ Training interrupted by user (KeyboardInterrupt).")
    # Handle interruption gracefully: Save current state if possible
    emergency_adapter_path = os.path.join(output_dir, "interrupt_checkpoint_adapter")
    print(f"Attempting to save current adapter state due to interruption...")
    try:
        if hasattr(model, 'save_pretrained'): # Check if model has the method (it should)
             model.save_pretrained(emergency_adapter_path)
             tokenizer.save_pretrained(emergency_adapter_path)
             trainer.save_state() # Also save trainer state if possible
             print(f"Adapter, tokenizer, and trainer state saved to {emergency_adapter_path}")
        else:
             print("Model object does not support 'save_pretrained'. Cannot save adapter.")
    except Exception as save_e:
        print(f"Failed to save emergency adapter/state: {save_e}")

except Exception as e:
    print(f"\n❌ An error occurred during training: {e}")
    # Save emergency adapter on other errors too
    emergency_adapter_path = os.path.join(output_dir, "error_checkpoint_adapter")
    print(f"Attempting to save adapter state due to error...")
    try:
        if hasattr(model, 'save_pretrained'):
             model.save_pretrained(emergency_adapter_path)
             tokenizer.save_pretrained(emergency_adapter_path)
             trainer.save_state()
             print(f"Adapter, tokenizer, and trainer state saved to {emergency_adapter_path}")
        else:
             print("Model object does not support 'save_pretrained'. Cannot save adapter.")
    except Exception as save_e:
        print(f"Failed to save emergency adapter/state after error: {save_e}")
    # Re-raise the original exception after attempting to save
    import traceback
    traceback.print_exc() # Print detailed traceback
    # raise e # Optional: re-raise if you want the script to exit with error status

finally:
    # Clean up GPU memory regardless of success or failure
    print("\nCleaning up GPU memory...")
    del model
    del trainer
    # del train_dataset # Optional: delete datasets if large
    # del validation_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\nTraining script finished.")

<ipython-input-2-fc1dff8ff4c9>:16: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU does not support BF16 or is older, using float16.
Initializing Unsloth FastLanguageModel: unsloth/Qwen2.5-Math-1.5B-bnb-4bit...
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 6.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 671 | Num Epochs = 5 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768/5,000,000,000 (0.37% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.938400,1.256075
2,1.005800,1.252345
3,0.969100,1.249908
4,1.002500,1.248900


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ Training completed successfully!
Total Training Time: 2:47:51
Saving final LoRA adapter to ./qwen_math_nbody_lora_unsloth_continued_pt/final_adapter...
Adapter and tokenizer saved successfully to ./qwen_math_nbody_lora_unsloth_continued_pt/final_adapter
***** train metrics *****
  total_flos               = 48418994GF
  train_loss               =     1.0002
  train_runtime            = 2:47:49.66
  train_samples_per_second =      0.333
  train_steps_per_second   =       0.01

Cleaning up GPU memory...

Training script finished.
